# Throttled Job Submission: `throttle_submit.py`

## The One-Liner

```bash
./throttle_submit.py --throttle 4 -- ./vsim.py runsim \
    --slurm-override nice 100 \
    --ants 0~9 \
    --channels 227~316 \
    --sky-model eor-grf-256 \
    --sky-realization rlzn_seed_222_offsetfix_freqslic_grf \
    --n-time-chunks 288 \
    --do-time-chunks 0~48 \
    --simulator matvis \
    --beam-map-csv beam_map_10_80MHzref_gauss_nonspectral_skysd111_eoroffsetfix.csv \
    --beamvar-type airyred \
    --rerun-existing
```

This does **exactly** what `vsim.py runsim` would do, but limits to 4 concurrent jobs.

---

## What It Does Internally

1. Appends `--dry-run` to your `vsim.py` command → generates all 4,272 batch scripts
2. Captures `modeldir is ...` from vsim's stdout → locates the job-specific subfolder
3. Collects the `fch*` scripts from **that folder only**
4. Reads the first script's `#SBATCH` lines (partition, mem, time, nice, etc.)
5. Writes a SLURM array wrapper that replays each script identically
6. Submits the wrapper with `sbatch`

**Nothing changes** about the actual jobs — same resources, same log paths,
same `hera-sim-vis.py` commands. The only addition is `%N` throttling.

---

## Usage

```bash
# Basic usage (4 concurrent jobs):
./throttle_submit.py --throttle 4 -- ./vsim.py runsim [your flags]

# More aggressive (16 concurrent):
./throttle_submit.py --throttle 16 -- ./vsim.py runsim [your flags]

# Preview without submitting:
./throttle_submit.py --throttle 4 --dry-run -- ./vsim.py runsim [your flags]
```

The `--` separates throttle_submit's flags from vsim's flags.

### Changing throttle after submission

```bash
scontrol update ArrayTaskThrottle=8 JobId=<JOB_ID>
```

---

In [ ]:
# Show the throttle_submit.py script
from pathlib import Path
script = Path('/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/throttle_submit.py')
print(script.read_text())

## How the Wrapper Replays Each Script

For array task N:

1. Reads line N from `batch_scripts/vsim_job_list.txt` → gets path to a specific `fch0227_chunk00005` script
2. Extracts `--output=` from that script's `#SBATCH` header → original log path
3. Extracts `--job-name=` → updates this task's name via `scontrol`
4. Runs: `bash "${SCRIPT}" > "${ORIG_OUTPUT}" 2>&1`

The inner script's `#SBATCH` lines are comments under `bash`. The real
commands (conda activate, module load, hera-sim-vis.py) execute inside
the wrapper's SLURM allocation.

---

## Why This Approach

`vsim.py runsim` calls `sbatch` 4,272 times in a for-loop. There's no way
to throttle 4,272 independent jobs after the fact.

SLURM's `--array=0-4271%4` is the only way to throttle — but it requires
**one** `sbatch` call creating all tasks. So:

1. `--dry-run` makes vsim **write** the scripts but **skip** `sbatch`
2. `throttle_submit.py` collects those scripts and submits them as one array job
3. Each array task runs one script via `bash`, inside the throttle

---

## Monitoring

| Command | Purpose |
|---------|---------|
| `squeue -u $USER` | All your jobs |
| `squeue -j <ID>` | Array job status |
| `scancel <ID>` | Cancel entire array |
| `scancel <ID>_[10-20]` | Cancel tasks 10–20 only |
| `scontrol update ArrayTaskThrottle=8 JobId=<ID>` | Change throttle live |
| `sacct -j <ID> --format=JobID,Elapsed,State,MaxRSS` | Post-run summary |

---

## Key Details

### `#SBATCH` source chain

```
hpc-configs/NRAO.yaml  →  --slurm-override  →  run_sim.py auto (job-name, output, time)
       (defaults)           (your CLI)              (per-job)
```

The wrapper copies the resource lines (partition, mem, time, nice) verbatim
from the generated scripts. Per-job fields (job-name, output) are extracted
at runtime for each array task.

### modeldir auto-detection

`run_sim.py` prints `modeldir is <path>` during dry-run. `throttle_submit.py`
captures this to find the exact subfolder under `batch_scripts/vis/` for
this specific job. No manual path pasting needed.

---

## Files Created

| File | Purpose |
|------|---------|
| `batch_scripts/vis/<modeldir>/fch*` | Generated by `vsim.py --dry-run` (one per channel×chunk) |
| `batch_scripts/vsim_job_list.txt` | Absolute paths to all scripts for this job |
| `batch_scripts/vsim_throttled.sbatch` | The array wrapper submitted to SLURM |